# Entregable Nro. 1: Análisis y Visualización de Sitios Web para Detección de Fraudes

Este notebook contiene la resolución correspondiente al primer trabajo: análisis exploratorio de datos, para el proyecto "El Robo del Siglo → Digital", enfocado en la detección de sitios web fraudulentos.

**Grupo 1:** Manuel Lopez Werlen - Ayelen Margarita Bertorello - Silvio Fabian Marasca - Ignacio Ariel Lopez Parra

Para comenzar realizamos las siguientes aclaraciones: Como equipo/grupo decidimos trabajar con el archivo compartido desde la mentora, para realizar nuestra versión lo modificamos e intervenimos la notebook, en base a las siguientes tareas:
1. Configuración del Entorno y Generar un dataset desde el sitio web: Tranco List
2. Procesamiento del dataset
3. Análisis de forma exploratoria del dataset (EDA)
4. Scraping general
5. Visualización los datos
6. Identificación de datos sospechosos
7. Generación de datos sintéticos
8. Conclusiones y pasos a futuro
8. Generación de un dataset "Final"

#1. Configuración del Entorno y Generar un dataset desde el sitio web: Tranco-List

In [1]:
!pip install requests pandas fake_useragent tldextract python-whois Faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.4/107.4 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.2/104.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 18.1 MB/s eta 0:00:00


In [2]:
import pandas as pd
import requests
import tldextract
import whois
import random
import re
from fake_useragent import UserAgent
from faker import Faker
from datetime import datetime
from tqdm.notebook import tqdm
from urllib.parse import urlparse
import time
import ssl
import socket
import warnings
from bs4 import BeautifulSoup
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dateutil import parser


warnings.filterwarnings("ignore")

fake = Faker('es_AR')
ua = UserAgent()

tqdm.pandas()

Generar un dataset desde el sitio web Tranco-List

In [4]:
# Descargamos el top millon de sitios desde Tranco-List
url = "https://tranco-list.eu/download/X456N/1000000"
df = pd.read_csv(url, header=None, names=["rank", "domain"])
# Filtramos para quedarnos solo con los sitios argentinos
df_ar = df[df["domain"].str.endswith(".ar")].reset_index(drop=True)

#df_ar = pd.read_csv("sitios_legitimos.csv")

#2. Procesamiento del dataset

In [5]:
# Definimos una función para categorizar los dominios
def clasificar_categoria(dominio):
    """
    Clasifica un dominio web en una categoría temática general según palabras clave presentes.

    Esta función busca términos genéricos en el nombre del dominio para asignarle una categoría
    temática amplia como "noticias", "gobierno", "banca", "e-commerce" o "educacion". Si no
    encuentra coincidencias, clasifica el dominio como "otro".

    Args:
        dominio (str): Nombre de dominio (por ejemplo, "noticiasargentinas.com.ar").

    Returns:
        str: Categoría general a la que pertenece el dominio. Puede ser:
            - "noticias"
            - "gobierno"
            - "banca"
            - "e-commerce"
            - "educacion"
            - "otro"
    """
    dominio = dominio.lower()

    if any(x in dominio for x in ["news", "noticia", "diario", "prensa", "periodico", "press"]):
        return "noticias"
    elif any(x in dominio for x in ["gob", "gov", "municipio", "ministerio", "provincia", ".gob.", ".gov."]):
        return "gobierno"
    elif any(x in dominio for x in ["banco", "bank", "finance", "finanzas", "credito", "loan", "tarjeta"]):
        return "banca"
    elif any(x in dominio for x in ["shop", "store", "tienda", "ecommerce", "comprar", "venta", "oferta"]):
        return "e-commerce"
    elif any(x in dominio for x in ["edu", "universidad", "facultad", "campus", "colegio", "escuela", "instituto"]):
        return "educacion"
    else:
        return "otro"

In [6]:

def procesar_dominio_basico(dominio: str) -> dict:
    """
    Procesa información básica y WHOIS de un dominio, generando las features correspondientes.
    """

    url = f"http://{dominio}"
    parsed = urlparse(url)
    hostname = parsed.hostname or dominio

    # --- Métricas estáticas de la URL ---
    url_length = len(url)
    num_dashes = dominio.count('-')
    num_digits = sum(c.isdigit() for c in dominio)
    num_special_chars = len(re.findall(r'[^\w\s:/.-]', dominio))
    num_dots = dominio.count('.')
    num_underscores = dominio.count('_')
    num_dashes_in_hostname = hostname.count('-')
    double_slash_in_path = '//' in parsed.path if parsed.path else False

    path_segments = len(parsed.path.strip('/').split('/')) if parsed.path else 0
    hostname_length = len(hostname)
    path_length = len(parsed.path)
    query_length = len(parsed.query)

    # --- Extraer TLD ---
    ext = tldextract.extract(dominio)
    tld = ext.suffix

    # --- WHOIS ---
    creation_date_iso = None
    expiration_date_iso = None
    registrar = None
    country_registered = None
    site_age_years = None
    time_to_expire_years = None
    registration_time = None

    try:
        socket.setdefaulttimeout(5)  # evitar bloqueos WHOIS
        info_whois = whois.whois(dominio)

        # Normalizar fechas
        creation = info_whois.creation_date
        expiration = info_whois.expiration_date

        if isinstance(creation, list):
            creation = creation[0]
        if isinstance(expiration, list):
            expiration = expiration[0]

        # Convertir strings a datetime
        if isinstance(creation, str):
            try:
                creation = parser.parse(creation)
            except:
                creation = None
        if isinstance(expiration, str):
            try:
                expiration = parser.parse(expiration)
            except:
                expiration = None

        # Guardar en ISO si disponibles
        creation_date_iso = creation.isoformat() if creation else None
        expiration_date_iso = expiration.isoformat() if expiration else None

        # Calcular métricas temporales
        if creation:
            site_age_years = round((datetime.now() - creation).days / 365, 2)
        if expiration:
            time_to_expire_years = round((expiration - datetime.now()).days / 365, 2)
        if creation and expiration:
            registration_time = round((expiration - creation).days / 365, 2)

        registrar = info_whois.registrar
        country_registered = (
            info_whois.get("country") or
            info_whois.get("registrant_country") or
            "Desconocido"
        )
    except Exception:
        pass

    # --- Flag si está registrado en Argentina ---
    is_registered_in_ar = bool(country_registered and "argentina" in country_registered.lower())

    return {
        # Identificación
        "url": url,
        "tld": tld,
        "is_phishing": None,  # Placeholder para etiquetado posterior

        # Estructura URL
        "url_length": url_length,
        "num_dashes": num_dashes,
        "num_digits": num_digits,
        "num_special_chars": num_special_chars,
        "path_segments": path_segments,
        "num_dots": num_dots,
        "num_underscores": num_underscores,
        "num_dashes_in_hostname": num_dashes_in_hostname,
        "double_slash_in_path": double_slash_in_path,
        "hostname_length": hostname_length,
        "path_length": path_length,
        "query_length": query_length,

        # WHOIS y temporalidad
        "registration_time": registration_time,
        "creation_date": creation_date_iso,
        "expiration_date": expiration_date_iso,
        "site_age_years": site_age_years,
        "time_to_expire_years": time_to_expire_years,
        "registrar": registrar,
        "country_registered": country_registered,
        "is_registered_in_ar": is_registered_in_ar
    }


In [10]:
# Limitar (por ahora) a los primeros 100 para evitar bloqueos
subset = df_ar

# Obtener Datos
results = []
for row in tqdm(subset.itertuples(index=False), total=len(subset), desc="Procesando dominios"):
    info = procesar_dominio_basico(row.domain)
    info["is_phishing"] = False  # Agregar la columna con valor True
    results.append(info)
    time.sleep(1)

# Guardar CSV
df_procesado = pd.DataFrame(results)
df_procesado.to_csv("sitios_argentinos_procesados.csv", index=False)
print("✅ CSV enriquecido guardado como sitios_argentinos_procesados.csv")
#df_procesado = pd.read_csv("sitios_argentinos_procesados.csv")

Procesando dominios:   0%|          | 0/3307 [00:00<?, ?it/s]

KeyboardInterrupt: 

**Función para obtener lista de sitios fraudulentos.**

Decidimos nutrirnos de: OpenPhish un servicio que ofrece una lista pública y actualizada de sitios web de phishing.
Este sitio es automático, actualizado y gratuito, proporciona casos reales en circulación y nos ayuda a no depender solo de sitios simulados para estudiar fraudes digitales.



In [ ]:
# URL del feed de OpenPhish (gratuito, sin autenticación)

url_feed = "https://openphish.com/feed.txt"

# Descargar el contenido del feed
response = requests.get(url_feed)

# Verificar si la descarga fue exitosa
if response.status_code == 200:
    urls = response.text.strip().split('\n')

    # Crear DataFrame
    df_phishing = pd.DataFrame(urls, columns=["url"])

    # Guardar como CSV
    df_phishing.to_csv("sitios_fraudulentos.csv", index=False)
    print("✅ Archivo guardado como 'sitios_fraudulentos.csv'")
else:
    print(f"❌ Error al descargar el feed: código {response.status_code}")


#3. Análisis de forma exploratoria del dataset (EDA)

In [ ]:
# Estadísticas descriptivas de variables
print("Estadísticas descriptivas de variables numéricas:")
numeric_stats = df_procesado.describe(include=[np.number])
numeric_stats

In [ ]:
# Estadísticas de variables categóricas
print("\nEstadísticas de variables categóricas:")
categorical_stats = df_procesado.describe(include=['object'])
categorical_stats

In [ ]:
# Verificar valores nulos
print("Valores nulos por columna:")
null_counts = df_procesado.isnull().sum()
null_counts

#4. Scraping general

En esta sección vamos a implementar un proceso de web scraping básico sobre los dominios recopilados, con el objetivo de obtener información dinámica directamente desde los sitios web. Esta información permite enriquecer el dataset con características que no están disponibles mediante fuentes estáticas, y aporta contexto sobre el comportamiento real de los sitios analizados.

In [ ]:

def enriquecer_dominio_scraping(dominio: str) -> dict:
    """
    Obtiene datos dinámicos del sitio mediante requests y análisis HTML.
    """
    esquemas = ["https", "http"]
    titulo = ""
    tiempo_respuesta = None
    responde = False
    codigo_estado = None
    url_redireccionada = None
    tiene_https = False
    tiene_ssl = False
    meta_keywords = ""
    iframe_present = False
    insecure_forms = False
    submit_info_to_email = False
    abnormal_form_action = False
    html_text = ""

    for esquema in esquemas:
        try:
            url = f"{esquema}://{dominio}"
            inicio = time.time()
            r = requests.get(url, timeout=6, allow_redirects=True, headers={
                "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/117.0 Safari/537.36"
            })
            fin = time.time()

            tiempo_respuesta = round(fin - inicio, 3)
            responde = True
            codigo_estado = r.status_code
            url_redireccionada = r.url
            html_text = r.text

            soup = BeautifulSoup(html_text, "html.parser")
            titulo_tag = soup.find("title")
            if titulo_tag:
                titulo = titulo_tag.text.strip()
            if esquema == "https":
                tiene_https = True
            break
        except requests.exceptions.RequestException:
            continue

    soup = BeautifulSoup(html_text, "html.parser") if html_text else None

    # --- Title y meta keywords ---
    longitud_titulo = len(titulo)
    if soup:
        meta_tag = soup.find("meta", attrs={"name": "keywords"})
        if meta_tag and "content" in meta_tag.attrs:
            meta_keywords = meta_tag["content"].lower()

    # --- SSL check ---
    if tiene_https:
        try:
            hostname = urlparse(url).hostname
            context = ssl.create_default_context()
            with socket.create_connection((hostname, 443), timeout=5) as sock:
                with context.wrap_socket(sock, server_hostname=hostname) as ssock:
                    tiene_ssl = bool(ssock.getpeercert())
        except:
            pass

    # --- Analizar HTML para iframes y forms ---
    if soup:
        iframe_present = bool(soup.find("iframe"))
        forms = soup.find_all("form")
        for form in forms:
            action = form.get("action", "").lower()
            if not action or action.startswith("http://"):
                insecure_forms = True
            if "mailto:" in action:
                submit_info_to_email = True
            # Acción que apunta a otro dominio
            if action.startswith("http"):
                action_host = urlparse(action).hostname
                if action_host and not action_host.endswith(dominio):
                    abnormal_form_action = True

    # --- Indicadores engañosos ---
    sensitive_words = ["login", "secure", "account", "bank", "verify", "update"]
    sensitive_words_count = sum(titulo.lower().count(word) for word in sensitive_words)
    https_in_hostname = "https" in dominio
    random_string = bool(re.search(r"[a-z]{5,}\d{3,}|[0-9]{5,}", dominio))

    bancos = [
        "santander", "bbva", "galicia", "banco nación", "bna", "hipotecario", "provincia",
        "macro", "comafi", "brubank", "itau", "supervielle", "patagonia"
    ]
    entidades_publicas = [
        "afip", "anses", "anmat", "ministerio", "gobierno", "municipio", "secretaria", "renaper",
        "dgr", "dine", "senado", "diputados", "presidencia"
    ]
    redes_sociales = [
        "facebook", "instagram", "twitter", "whatsapp", "tiktok", "linkedin", "telegram", "snapchat"
    ]
    empresas_y_servicios = [
        "mercadolibre", "mercadopago", "despegar", "globant", "tenaris", "ypf", "rappi", "pedidosya",
        "todoPago", "naranja", "ripley", "uala", "plin", "cencosud"
    ]
    # Unificar listas y normalizar dominio
    marcas_populares = bancos + entidades_publicas + redes_sociales + empresas_y_servicios
    dominio_lower = dominio.lower()

    embedded_brand_name = any(brand.lower() in dominio_lower for brand in marcas_populares)

    domain_in_subdomains = False
    domain_in_paths = False
    parsed_url = urlparse(url_redireccionada or url)
    base_domain = dominio.split(".")[0]
    if base_domain in parsed_url.netloc and parsed_url.netloc != dominio:
        domain_in_subdomains = True
    if base_domain in parsed_url.path:
        domain_in_paths = True

    categoria = clasificar_categoria(titulo or dominio)

    return {
        # Seguridad
        "has_https": tiene_https,
        "has_ssl_cert": tiene_ssl,
        "iframe_present": iframe_present,
        "insecure_forms": insecure_forms,
        "submit_info_to_email": submit_info_to_email,
        "abnormal_form_action": abnormal_form_action,

        # Respuesta servidor
        "response_time": tiempo_respuesta,
        "responds": responde,
        "http_status_code": codigo_estado,
        "redirected_url": url_redireccionada,

        # Contenido
        "title": titulo,
        "title_length": longitud_titulo,
        "meta_keywords": meta_keywords,
        "category": categoria,

        # Indicadores engañosos
        "random_string": random_string,
        "sensitive_words_count": sensitive_words_count,
        "embedded_brand_name": embedded_brand_name,
        "https_in_hostname": https_in_hostname,
        "domain_in_subdomains": domain_in_subdomains,
        "domain_in_paths": domain_in_paths
    }


In [ ]:
# Lista para almacenar los nuevos datos
datos_scraping = []

for row in tqdm(df_procesado.itertuples(index=False), total=len(df_procesado), desc="Enriqueciendo con scraping"):
    dominio = row.url.replace("http://", "").replace("https://", "").split("/")[0]
    scraping_info = enriquecer_dominio_scraping(dominio)
    datos_scraping.append(scraping_info)
    time.sleep(1)

# Convertir la lista en DataFrame
df_scraping = pd.DataFrame(datos_scraping)

# Combinar con el DataFrame original
df_final = pd.concat([df_procesado.reset_index(drop=True), df_scraping], axis=1)

# Guardar CSV enriquecido
df_final.to_csv("sitios_argentinos_enriquecidos.csv", index=False)
print("✅ CSV enriquecido guardado como sitios_argentinos_enriquecidos.csv")

#5. Visualización de datos

In [ ]:
# Distribución de TLDs
plt.figure(figsize=(10, 6))
tld_counts = df_final['tld'].value_counts()
sns.barplot(x=tld_counts.index, y=tld_counts.values)
plt.title('Distribución de Dominios por TLD')
plt.xlabel('TLD')
plt.ylabel('Cantidad de Sitios')
plt.tight_layout()
plt.show()

_De los sitios observados, solo el 18% presenta TLD diferentes de .com.ar, edu.ar o gob.ar. Estos sitios presentan mas posibilidades de ser phising y deberían ser analizados en mayor profundidad_

In [ ]:
# Histograma fecha de creación
df_final['creation_date'] = pd.to_datetime(df_final['creation_date'], errors='coerce')
plt.figure(figsize=(10, 4))
sns.histplot(df_final['creation_date'].dropna(), bins=10, kde=False)
plt.title('Distribución de fechas de creación')
plt.xlabel('Fecha de creación')
plt.ylabel('Cantidad')
plt.xticks(rotation=45)
plt.show()

_La mayoría de los sitios analizados fueron creados entre septiembre y noviembre de 2013. Ninguno de los sitios top .ar fue creado en los últimos diez años, lo cual tiene sentido dada su legitimidad/popularidad._

In [ ]:
# Proporción de variable tiene_ssl
ssl_counts = df_final['has_ssl_cert'].value_counts()
labels = ['Con SSL' if val else 'Sin SSL' for val in ssl_counts.index]
plt.figure(figsize=(7, 7))
plt.pie(ssl_counts, labels=labels, autopct='%1.1f%%', startangle=140)
plt.title('Proporción de Sitios con Certificado SSL')
plt.tight_layout()
plt.show()

_Aquí podemos observar que casi el 80% de los sitios reales analizados utilizan SSL. Si bien el hecho de que no se use no es un indicador de que un sitio sea efectivamente de phising, su no utilización puede ser un indicador._

In [ ]:
# Información de las categorías agregadas
df_final["category"].value_counts()

In [ ]:
# Distribución de sitios por categoría
plt.figure(figsize=(10, 6))
category_counts = df_final['category'].value_counts()
sns.barplot(x=category_counts.index, y=category_counts.values)
plt.title('Distribución de Sitios por Categoría')
plt.xlabel('Categoría')
plt.ylabel('Cantidad de Sitios')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

_Se analizaron los dominios para categorizar los sitios. La búsqueda debe refinarse más pero es útil esta clasificación para poder determinar si los sitios son sospechosos o no._

In [ ]:
# Distribución de tiempos de respuesta
plt.figure(figsize=(10, 6))
sns.histplot(df_final['response_time'], kde=True)
plt.title('Distribución de Tiempos de Respuesta')
plt.xlabel('Tiempo de Respuesta (segundos)')
plt.ylabel('Frecuencia')
plt.tight_layout()
plt.show()

_En este gráfico podemos ver que la mayoría de los sitios responde en tiempos  menores o iguales a un segundo, y un porcentaje muy bajo, por encima de los dos segundos. Un sitio con un tiempo de respuesta mayor, puede ser tenido en cuenta como sospechoso._

#6. Identificación de datos sospechosos

In [ ]:
# Dominios recientes (creados hace menos de 1 año)
dominios_recientes = df_final[(df_final['registration_time'] > 0) & (df_final['registration_time'] < 1)]
total_dominios = len(df_final)
cantidad_recientes = len(dominios_recientes)
porcentaje = (cantidad_recientes / total_dominios) * 100
print(f"Dominios con tiempo de registro menor a 1 año: {porcentaje:.2f}% ({cantidad_recientes} de {total_dominios})")

In [ ]:
# URLs sospechosas por alto número de guiones o caracteres especiales
urls_sospechosas = df_final[(df_final['num_dashes'] > 3) | (df_final['num_special_chars'] > 3)]
total_dominios = len(df_final)
cantidad_sospechosas = len(urls_sospechosas)
porcentaje = (cantidad_sospechosas / total_dominios) * 100
print(f"URLs con alta cantidad de guiones o caracteres especiales: {porcentaje:.2f}% ({cantidad_sospechosas} de {total_dominios})")

In [ ]:
# TLDs inusuales (no son .com.ar, .edu.ar o .gob.ar)
tlds_inusuales = df_final[~df_final['tld'].isin(['com.ar', 'edu.ar', 'gob.ar'])]
total_dominios = len(df_final)
cantidad_tld_inusual = len(tlds_inusuales)
porcentaje = (cantidad_tld_inusual / total_dominios) * 100
print(f"TLDs inusuales (distintos de .com.ar, .edu.ar o .gob.ar): {porcentaje:.2f}% ({cantidad_tld_inusual} de {total_dominios})")

In [ ]:
# Dominios sin certificado SSL
sin_ssl = df_final[df_final['has_ssl_cert'] == False]
total_dominios = len(df_final)
cantidad_sin_ssl = len(sin_ssl)
porcentaje = (cantidad_sin_ssl / total_dominios) * 100
print(f"Dominios sin certificado SSL: {porcentaje:.2f}% ({cantidad_sin_ssl} de {total_dominios})")

In [ ]:
# Dominios con palabras clave sospechosas
palabras_sospechosas = [
    'banco', 'seguridad', 'actualización', 'verificación', 'urgente',
    'cuenta', 'tarjeta', 'débito', 'crédito', 'cbu', 'homebanking', 'transferencia',
    'validar', 'confirmación', 'clave', 'contraseña', 'acceso', 'autenticación', 'identidad',
    'importante', 'alerta','seguridad', 'aviso', 'inmediato', 'emergencia', 'notificación','ingresos', 'oficial',
    'regalo', 'sorteo', 'premio', 'ganaste', 'oferta', 'gratis', 'promoción', 'exclusivo',
    'ingresar', 'activar', 'acceder', 'desbloquear', 'iniciar sesión', 'confirmar'
]

# Función para detectar palabras en la URL
def contiene_palabra_sospechosa(texto):
    if pd.isna(texto):
        return False
    texto = texto.lower()
    return any(palabra in texto for palabra in palabras_sospechosas)

# Aplicar la función
urls_palabras_clave = df_final[df_final['url'].apply(contiene_palabra_sospechosa)]
total_dominios = len(df_final)
cantidad_con_palabras = len(urls_palabras_clave)
porcentaje = (cantidad_con_palabras / total_dominios) * 100
print(f"URLs con palabras clave sospechosas: {porcentaje:.2f}% ({cantidad_con_palabras} de {total_dominios})")

In [ ]:
# Buscar palabras clave sospechosas

# Función para contar coincidencias en el título
def analizar_texto_sospechoso(texto):
    if pd.isna(texto):
        return 0
    texto_lower = texto.lower()
    return sum(1 for palabra in palabras_sospechosas if palabra in texto_lower)

# Aplicar sin modificar el DataFrame
coincidencias = df_final['title'].apply(analizar_texto_sospechoso)
titulos_sospechosos = df_final[coincidencias > 0]

# Imprimir resumen
total_dominios = len(df_final)
cantidad_sospechosos = len(titulos_sospechosos)
porcentaje = (cantidad_sospechosos / total_dominios) * 100

print(f"Títulos con palabras clave sospechosas: {porcentaje:.2f}% ({cantidad_sospechosos} de {total_dominios})")

Se aplicaron distintos criterios de detección sobre el subconjunto de 100 sitios reales para identificar posibles patrones asociados al comportamiento de sitios fraudulentos.
Si bien no se encontraron dominios con registro reciente ni URLs con estructuras sospechosas (guiones o caracteres especiales en exceso), sí se detectaron algunos indicios a tener en cuenta: un 18% presenta TLDs inusuales, el 22% carece de certificado SSL, y entre el 3% y el 6% de los sitios contienen palabras clave sospechosas tanto en la URL como en el título. Estos indicadores, si bien no implican necesariamente phishing, pueden ser señales de alerta en un análisis más amplio o en combinación con otros factores.

#7. Generación de datos sintéticos

In [ ]:
# Función para generar datos sinteticos
def generar_datos_sinteticos(n=100):
    datos_sinteticos = []

    for _ in range(n):
        sitio = fake.domain_name()
        titulo = fake.sentence(nb_words=random.randint(3, 10))
        tiempo_respuesta = round(random.uniform(0.1, 2.5), 3)
        responde = random.choice([True, False])
        codigo_estado = random.choice([200, 301, 302, 403, 404, 500])
        url_redireccionada = sitio if codigo_estado in [200, 403, 404, 500] else f"{sitio}/redirected"

        tld_info = tldextract.extract(sitio)
        tld = tld_info.suffix

        longitud_url = len(sitio)
        tiene_https = random.choice([True, False])
        tiene_ssl = tiene_https and random.choice([True, False])
        cantidad_guiones = sitio.count('-')
        cantidad_digitos = sum(c.isdigit() for c in sitio)
        caracteres_especiales = len(re.findall(r'[^\w\s:/.-]', sitio))
        segmentos_path = len(urlparse(sitio).path.strip('/').split('/')) if urlparse(sitio).path else 0
        longitud_titulo = len(titulo)
        palabras_clave = len(titulo.split())
        tiempo_registro = random.randint(1, 20)

        fecha_creacion = fake.date_between(start_date='-20y', end_date='-1y')
        fecha_expiracion = fake.date_between(start_date=fecha_creacion, end_date='+5y')

        registrador = fake.company()
        pais_registro = fake.country()

        categoria = clasificar_categoria(sitio)

        datos_sinteticos.append({
            "url": sitio,
            "titulo": titulo,
            "tiempo_respuesta": tiempo_respuesta,
            "responde": responde,
            "codigo_estado": codigo_estado,
            "url_redireccionada": url_redireccionada,
            "categoria": categoria,
            "longitud_url": longitud_url,
            "tiene_https": tiene_https,
            "tiene_ssl": tiene_ssl,
            "cantidad_guiones": cantidad_guiones,
            "cantidad_digitos": cantidad_digitos,
            "caracteres_especiales": caracteres_especiales,
            "segmentos_path": segmentos_path,
            "tld": tld,
            "longitud_titulo": longitud_titulo,
            "palabras_clave": palabras_clave,
            "tiempo_registro": tiempo_registro,
            "fecha_creacion": fecha_creacion.isoformat(),
            "fecha_expiracion": fecha_expiracion.isoformat(),
            "registrador": registrador,
            "pais_registro": pais_registro,
            "es_phishing": None
        })

    return pd.DataFrame(datos_sinteticos)


In [ ]:
df_sintetico = generar_datos_sinteticos(100)
df_sintetico.to_csv("sitios_argentinos_sinteticos.csv", index=False)
print("✅ CSV sintetico guardado como sitios_argentinos_sinteticos.csv")

#8. Conclusiones y pasos a futuro

En este trabajo se desarrolló un flujo completo para la recopilación y procesamiento de dominios web argentinos, enriqueciéndolo con el objetivo de generar un dataset estructurado útil para entrenar un modelo de clasificación supervisada. Se implementaron funciones eficientes para extraer características estáticas de los dominios (como longitud, cantidad de guiones, TLD, etc.) así como funciones de scraping más costosas para obtener información dinámica como tiempos de respuesta, redirecciones y títulos HTML.

Además, se generaron datos sintéticos simulando sitios legítimos y se obtuvieron datos de sitios de phishing reales. Se incluyó información WHOIS, como país de registro, registrador y fechas de creación y expiración del dominio. Finalmente, se unificaron todos los datos en un conjunto enriquecido y etiquetado, exportado a CSV para su posterior análisis.


In [ ]:
# Realizamos un resumen de lo anterior con ciertas aclaraciones que pensamos pueden ser útiles a futuro, como así también instrucciones de pasos a seguir.

print("Conclusiones:")
print("1. Se desarrolló un pipeline (un proceso automatizado y estructurado) completo para la recopilación, enriquecimiento y análisis de dominios web argentinos.")
print("2. La gran mayoría de los sitios argentinos analizados utilizan TLDs confiables como .com.ar, .edu.ar o .gob.ar.")
print("3. Los sitios que tiene mayor tiempo de respuesta puede ser tenidos en cuenta como fraude.")
print("4. La creacion del Dataset fue en base a 100 datos pero consideramos que para el siguimiento de este trabajo es necesario trabajar con todos los datos con los que contamos.")

print("\nPróximos Pasos (Entregable 2):")
print("1. Ampliar la recopilación de datos para incluir más sitios legítimos y fraudulentos.")
print("2. Realizar un análisis más profundo del contenido de los sitios, como así tambipen repensar nuevas categorías o subcategorías.")
print("3. Evaluar la implementación de técnicas de procesamiento de lenguaje natural para analizar el texto de los sitios.")

#9. Generación de Datasets Finales

Finalmente y a fines prácticos a continuación aclaramos cuales son los Datasets de salida:
Generamos los siguientes archivos CSV:
 - `sitios_argentinos_procesados.csv`: Datos extraídos de Tranco List
 - `sitios_fraudulentos.csv`: Datos extraídos de OpenPhish
 - `sitios_argentinos_enriquecidos.csv`: Datos enriquecidos
 - `sitios_argentinos_sinteticos.csv`:  Datos  sintéticos  generados con lobrería Faker